# 04.3 — Hypothesis 3

## Positive emotion and working-memory performance

### Hypothesis

Positive emotional induction improves working-memory performance
relative to neutral and negative emotional conditions.

### Primary comparisons

1. Positive versus Neutral
2. Positive versus Negative

### Primary outcomes

1. Trial-level response accuracy.
2. Reaction time among valid correct-response trials.

### Secondary analysis

The joint accuracy and reaction-time pattern is examined to determine
whether positive emotion improves overall processing efficiency.

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy import stats

from src import config as cfg
from src.utils import initialize_project

initialize_project()

## 3. Carregar os dados

In [ ]:
analysis = pd.read_csv(
    cfg.ANALYSIS_DATASET_FILE,
    low_memory=False,
)

print(f"Full dataset: {analysis.shape}")
print(
    "Participants:",
    analysis["participant_id"].nunique(),
)

## 4. Selecionar as três condições

<br>

Para H3 precisamos manter:

    - Positive
    - Neutral
    - Negative

porque os dois contrastes serão extraídos do mesmo conjunto.

In [ ]:
h3_data = (
    analysis.loc[
        analysis["emotion_condition"].isin(
            [
                "Positive",
                "Neutral",
                "Negative",
            ]
        )
    ]
    .copy()
)

print(f"H3 dataset: {h3_data.shape}")

print(
    h3_data["emotion_condition"]
    .value_counts()
)

In [ ]:
assert set(
    h3_data[
        "emotion_condition"
    ].dropna().unique()
) == {
    "Positive",
    "Neutral",
    "Negative",
}

# Auditoria da amostra

## 5. Participantes e trials

In [ ]:
h3_sample = (
    h3_data
    .groupby("emotion_condition")
    .agg(
        participants=(
            "participant_id",
            "nunique",
        ),
        trials=(
            "accuracy",
            "size",
        ),
        valid_accuracy=(
            "accuracy",
            "count",
        ),
    )
    .reset_index()
)

h3_sample

## 6. Participação nos contrastes

In [ ]:
participant_conditions_h3 = (
    h3_data[
        [
            "participant_id",
            "emotion_condition",
        ]
    ]
    .drop_duplicates()
)

condition_count_h3 = (
    participant_conditions_h3
    .groupby("participant_id")
    ["emotion_condition"]
    .nunique()
)

condition_count_h3.value_counts()

In [ ]:
positive_ids = set(
    h3_data.loc[
        h3_data[
            "emotion_condition"
        ] == "Positive",
        "participant_id",
    ]
)

neutral_ids = set(
    h3_data.loc[
        h3_data[
            "emotion_condition"
        ] == "Neutral",
        "participant_id",
    ]
)

negative_ids = set(
    h3_data.loc[
        h3_data[
            "emotion_condition"
        ] == "Negative",
        "participant_id",
    ]
)

print(
    "Positive and Neutral:",
    len(
        positive_ids
        & neutral_ids
    ),
)

print(
    "Positive and Negative:",
    len(
        positive_ids
        & negative_ids
    ),
)

# Accuracy

## 7. Estatísticas descritivas

In [ ]:
accuracy_summary_h3 = (
    h3_data
    .groupby("emotion_condition")
    .agg(
        participants=(
            "participant_id",
            "nunique",
        ),
        trials=(
            "accuracy",
            "size",
        ),
        correct=(
            "accuracy",
            "sum",
        ),
        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_sd=(
            "accuracy",
            "std",
        ),
        accuracy_median=(
            "accuracy",
            "median",
        ),
    )
    .reset_index()
)

accuracy_summary_h3

## 8. Accuracy participante × condição

In [ ]:
participant_accuracy_h3 = (
    h3_data
    .groupby(
        [
            "participant_id",
            "emotion_condition",
        ]
    )
    ["accuracy"]
    .mean()
    .reset_index()
)

participant_accuracy_h3.head()

## 9. Figura de accuracy

In [ ]:
accuracy_plot_h3 = (
    participant_accuracy_h3
    .groupby("emotion_condition")
    ["accuracy"]
    .agg(["mean", "sem"])
    .reindex(
        [
            "Neutral",
            "Negative",
            "Positive",
        ]
    )
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.errorbar(
    accuracy_plot_h3[
        "emotion_condition"
    ],
    accuracy_plot_h3[
        "mean"
    ],
    yerr=accuracy_plot_h3[
        "sem"
    ],
    marker="o",
    capsize=5,
)

ax.set_xlabel(
    "Emotional condition"
)

ax.set_ylabel(
    "Mean accuracy"
)

ax.set_ylim(
    0,
    1.05,
)

ax.set_title(
    "Accuracy across emotional conditions"
)

fig.tight_layout()

plt.show()

# Accuracy — Positive vs Neutral

## 10. GEE com Neutral como referência

In [ ]:
accuracy_formula_neutral_h3 = (
    "accuracy ~ "
    "C(emotion_condition, "
    "Treatment(reference='Neutral'))"
)

accuracy_model_neutral_h3 = (
    smf.gee(
        formula=(
            accuracy_formula_neutral_h3
        ),
        groups="participant_id",
        data=h3_data,
        family=(
            sm.families.Binomial()
        ),
        cov_struct=(
            sm.cov_struct.Exchangeable()
        ),
    )
    .fit()
)

print(
    accuracy_model_neutral_h3.summary()
)

## 11. Extrair tabela do modelo

In [ ]:
def extract_gee_results(
    result,
):
    confidence = (
        result.conf_int()
    )

    table = pd.DataFrame(
        {
            "term":
                result.params.index,

            "beta":
                result.params.values,

            "se":
                result.bse.values,

            "z":
                result.tvalues.values,

            "p":
                result.pvalues.values,

            "ci_low":
                confidence.iloc[
                    :, 0
                ].values,

            "ci_high":
                confidence.iloc[
                    :, 1
                ].values,
        }
    )

    table[
        "odds_ratio"
    ] = np.exp(
        table["beta"]
    )

    table[
        "or_ci_low"
    ] = np.exp(
        table["ci_low"]
    )

    table[
        "or_ci_high"
    ] = np.exp(
        table["ci_high"]
    )

    return table

In [ ]:
accuracy_results_neutral_h3 = (
    extract_gee_results(
        accuracy_model_neutral_h3
    )
)

accuracy_results_neutral_h3

In [ ]:
accuracy_positive_neutral_h3 = (
    accuracy_results_neutral_h3.loc[
        accuracy_results_neutral_h3[
            "term"
        ].str.contains(
            "Positive",
            regex=False,
        )
    ]
    .copy()
)

accuracy_positive_neutral_h3

# Accuracy — Positive vs Negative

## 12. GEE com Negative como referência

In [ ]:
accuracy_formula_negative_h3 = (
    "accuracy ~ "
    "C(emotion_condition, "
    "Treatment(reference='Negative'))"
)

accuracy_model_negative_h3 = (
    smf.gee(
        formula=(
            accuracy_formula_negative_h3
        ),
        groups="participant_id",
        data=h3_data,
        family=(
            sm.families.Binomial()
        ),
        cov_struct=(
            sm.cov_struct.Exchangeable()
        ),
    )
    .fit()
)

print(
    accuracy_model_negative_h3.summary()
)

In [ ]:
accuracy_results_negative_h3 = (
    extract_gee_results(
        accuracy_model_negative_h3
    )
)

accuracy_positive_negative_h3 = (
    accuracy_results_negative_h3.loc[
        accuracy_results_negative_h3[
            "term"
        ].str.contains(
            "Positive",
            regex=False,
        )
    ]
    .copy()
)

accuracy_positive_negative_h3

## 13. Consolidar os contrastes de accuracy

In [ ]:
accuracy_positive_neutral_h3[
    "contrast"
] = "Positive vs Neutral"

accuracy_positive_negative_h3[
    "contrast"
] = "Positive vs Negative"

accuracy_contrasts_h3 = pd.concat(
    [
        accuracy_positive_neutral_h3,
        accuracy_positive_negative_h3,
    ],
    ignore_index=True,
)

accuracy_contrasts_h3[
    [
        "contrast",
        "beta",
        "se",
        "z",
        "p",
        "odds_ratio",
        "or_ci_low",
        "or_ci_high",
    ]
]

# Reaction Time

## 14. Construir a base analítica

In [ ]:
rt_h3 = (
    h3_data.loc[
        (
            h3_data[
                "accuracy"
            ] == 1
        )
        &
        (
            h3_data[
                "rt_valid"
            ] == True
        )
    ]
    .copy()
)

rt_h3[
    "log_rt"
] = np.log(
    rt_h3[
        "reaction_time_ms"
    ]
)

print(
    "RT observations:",
    len(rt_h3),
)

print(
    "RT participants:",
    rt_h3[
        "participant_id"
    ].nunique(),
)

## 15. Estatísticas descritivas de RT

In [ ]:
rt_summary_h3 = (
    rt_h3
    .groupby("emotion_condition")
    .agg(
        participants=(
            "participant_id",
            "nunique",
        ),
        trials=(
            "reaction_time_ms",
            "size",
        ),
        mean_rt=(
            "reaction_time_ms",
            "mean",
        ),
        sd_rt=(
            "reaction_time_ms",
            "std",
        ),
        median_rt=(
            "reaction_time_ms",
            "median",
        ),
        mean_log_rt=(
            "log_rt",
            "mean",
        ),
    )
    .reset_index()
)

rt_summary_h3

## 16. RT participante × condição

In [ ]:
participant_rt_h3 = (
    rt_h3
    .groupby(
        [
            "participant_id",
            "emotion_condition",
        ]
    )
    .agg(
        mean_rt=(
            "reaction_time_ms",
            "mean",
        ),
        median_rt=(
            "reaction_time_ms",
            "median",
        ),
        mean_log_rt=(
            "log_rt",
            "mean",
        ),
    )
    .reset_index()
)

participant_rt_h3.head()

## 17. Figura de RT

In [ ]:
rt_plot_h3 = (
    participant_rt_h3
    .groupby("emotion_condition")
    ["mean_rt"]
    .agg(["mean", "sem"])
    .reindex(
        [
            "Neutral",
            "Negative",
            "Positive",
        ]
    )
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.errorbar(
    rt_plot_h3[
        "emotion_condition"
    ],
    rt_plot_h3[
        "mean"
    ],
    yerr=rt_plot_h3[
        "sem"
    ],
    marker="o",
    capsize=5,
)

ax.set_xlabel(
    "Emotional condition"
)

ax.set_ylabel(
    "Mean reaction time (ms)"
)

ax.set_title(
    "Reaction time across emotional conditions"
)

fig.tight_layout()

plt.show()

# RT — Positive vs Neutral

## 18. Mixed model com Neutral como referência

In [ ]:
rt_formula_neutral_h3 = (
    "log_rt ~ "
    "C(emotion_condition, "
    "Treatment(reference='Neutral'))"
)

rt_model_neutral_h3 = (
    smf.mixedlm(
        formula=(
            rt_formula_neutral_h3
        ),
        data=rt_h3,
        groups=rt_h3[
            "participant_id"
        ],
    )
)

rt_result_neutral_h3 = (
    rt_model_neutral_h3
    .fit(
        reml=False
    )
)

print(
    rt_result_neutral_h3.summary()
)

## 19. Função para extrair resultados MixedLM

In [ ]:
def extract_mixed_results(
    result,
):
    confidence = (
        result.conf_int()
    )

    table = pd.DataFrame(
        {
            "term":
                result.params.index,

            "beta":
                result.params.values,

            "se":
                result.bse.values,

            "z":
                result.tvalues.values,

            "p":
                result.pvalues.values,

            "ci_low":
                confidence.iloc[
                    :, 0
                ].values,

            "ci_high":
                confidence.iloc[
                    :, 1
                ].values,
        }
    )

    table[
        "rt_ratio"
    ] = np.exp(
        table["beta"]
    )

    table[
        "percent_change"
    ] = (
        np.exp(
            table["beta"]
        )
        - 1
    ) * 100

    table[
        "percent_ci_low"
    ] = (
        np.exp(
            table["ci_low"]
        )
        - 1
    ) * 100

    table[
        "percent_ci_high"
    ] = (
        np.exp(
            table["ci_high"]
        )
        - 1
    ) * 100

    return table

In [ ]:
rt_results_neutral_h3 = (
    extract_mixed_results(
        rt_result_neutral_h3
    )
)

rt_positive_neutral_h3 = (
    rt_results_neutral_h3.loc[
        rt_results_neutral_h3[
            "term"
        ].str.contains(
            "Positive",
            regex=False,
        )
    ]
    .copy()
)

rt_positive_neutral_h3

# RT — Positive vs Negative

## 20. Mixed model com Negative como referência

In [ ]:
rt_formula_negative_h3 = (
    "log_rt ~ "
    "C(emotion_condition, "
    "Treatment(reference='Negative'))"
)

rt_model_negative_h3 = (
    smf.mixedlm(
        formula=(
            rt_formula_negative_h3
        ),
        data=rt_h3,
        groups=rt_h3[
            "participant_id"
        ],
    )
)

rt_result_negative_h3 = (
    rt_model_negative_h3
    .fit(
        reml=False
    )
)

print(
    rt_result_negative_h3.summary()
)

In [ ]:
rt_results_negative_h3 = (
    extract_mixed_results(
        rt_result_negative_h3
    )
)

rt_positive_negative_h3 = (
    rt_results_negative_h3.loc[
        rt_results_negative_h3[
            "term"
        ].str.contains(
            "Positive",
            regex=False,
        )
    ]
    .copy()
)

rt_positive_negative_h3

## 21. Consolidar os contrastes de RT

In [ ]:
rt_positive_neutral_h3[
    "contrast"
] = "Positive vs Neutral"

rt_positive_negative_h3[
    "contrast"
] = "Positive vs Negative"

rt_contrasts_h3 = pd.concat(
    [
        rt_positive_neutral_h3,
        rt_positive_negative_h3,
    ],
    ignore_index=True,
)

rt_contrasts_h3[
    [
        "contrast",
        "beta",
        "se",
        "z",
        "p",
        "percent_change",
        "percent_ci_low",
        "percent_ci_high",
    ]
]

# Processing efficiency

## 20. Mixed model com Negative como referência

In [ ]:
rt_formula_negative_h3 = (
    "log_rt ~ "
    "C(emotion_condition, "
    "Treatment(reference='Negative'))"
)

rt_model_negative_h3 = (
    smf.mixedlm(
        formula=(
            rt_formula_negative_h3
        ),
        data=rt_h3,
        groups=rt_h3[
            "participant_id"
        ],
    )
)

rt_result_negative_h3 = (
    rt_model_negative_h3
    .fit(
        reml=False
    )
)

print(
    rt_result_negative_h3.summary()
)

In [ ]:
rt_results_negative_h3 = (
    extract_mixed_results(
        rt_result_negative_h3
    )
)

rt_positive_negative_h3 = (
    rt_results_negative_h3.loc[
        rt_results_negative_h3[
            "term"
        ].str.contains(
            "Positive",
            regex=False,
        )
    ]
    .copy()
)

rt_positive_negative_h3

## 21. Consolidar os contrastes de RT

In [ ]:
rt_positive_neutral_h3[
    "contrast"
] = "Positive vs Neutral"

rt_positive_negative_h3[
    "contrast"
] = "Positive vs Negative"

rt_contrasts_h3 = pd.concat(
    [
        rt_positive_neutral_h3,
        rt_positive_negative_h3,
    ],
    ignore_index=True,
)

rt_contrasts_h3[
    [
        "contrast",
        "beta",
        "se",
        "z",
        "p",
        "percent_change",
        "percent_ci_low",
        "percent_ci_high",
    ]
]

# Processing efficiency

## 22. Construir base participante × condição

In [ ]:
participant_performance_h3 = (
    participant_accuracy_h3
    .merge(
        participant_rt_h3[
            [
                "participant_id",
                "emotion_condition",
                "mean_rt",
                "median_rt",
            ]
        ],
        on=[
            "participant_id",
            "emotion_condition",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
participant_performance_h3[
    "inverse_efficiency"
] = (
    participant_performance_h3[
        "mean_rt"
    ]
    /
    participant_performance_h3[
        "accuracy"
    ]
)

participant_performance_h3.head()

## 23. Descrever eficiência

In [ ]:
efficiency_summary_h3 = (
    participant_performance_h3
    .groupby("emotion_condition")
    .agg(
        n=(
            "participant_id",
            "count",
        ),
        mean_accuracy=(
            "accuracy",
            "mean",
        ),
        mean_rt=(
            "mean_rt",
            "mean",
        ),
        mean_ies=(
            "inverse_efficiency",
            "mean",
        ),
        sd_ies=(
            "inverse_efficiency",
            "std",
        ),
    )
    .reset_index()
)

efficiency_summary_h3

# IES — Positive vs Neutral

## 24. Comparação pareada

In [ ]:
ies_wide_h3 = (
    participant_performance_h3
    .pivot_table(
        index="participant_id",
        columns="emotion_condition",
        values="inverse_efficiency",
        aggfunc="first",
    )
)

**Positive vs Neutral:**

In [ ]:
ies_positive_neutral = (
    ies_wide_h3[
        [
            "Positive",
            "Neutral",
        ]
    ]
    .dropna()
)

ies_test_positive_neutral = (
    stats.ttest_rel(
        ies_positive_neutral[
            "Positive"
        ],
        ies_positive_neutral[
            "Neutral"
        ],
    )
)

ies_test_positive_neutral

**Positive vs Negative:**

In [ ]:
ies_positive_negative = (
    ies_wide_h3[
        [
            "Positive",
            "Negative",
        ]
    ]
    .dropna()
)

ies_test_positive_negative = (
    stats.ttest_rel(
        ies_positive_negative[
            "Positive"
        ],
        ies_positive_negative[
            "Negative"
        ],
    )
)

ies_test_positive_negative

## 25. Consolidar IES

In [ ]:
ies_results_h3 = pd.DataFrame(
    [
        {
            "contrast":
                "Positive vs Neutral",

            "n":
                len(
                    ies_positive_neutral
                ),

            "positive_mean":
                ies_positive_neutral[
                    "Positive"
                ].mean(),

            "comparator_mean":
                ies_positive_neutral[
                    "Neutral"
                ].mean(),

            "mean_difference":
                (
                    ies_positive_neutral[
                        "Positive"
                    ]
                    -
                    ies_positive_neutral[
                        "Neutral"
                    ]
                ).mean(),

            "t":
                ies_test_positive_neutral.statistic,

            "p":
                ies_test_positive_neutral.pvalue,
        },
        {
            "contrast":
                "Positive vs Negative",

            "n":
                len(
                    ies_positive_negative
                ),

            "positive_mean":
                ies_positive_negative[
                    "Positive"
                ].mean(),

            "comparator_mean":
                ies_positive_negative[
                    "Negative"
                ].mean(),

            "mean_difference":
                (
                    ies_positive_negative[
                        "Positive"
                    ]
                    -
                    ies_positive_negative[
                        "Negative"
                    ]
                ).mean(),

            "t":
                ies_test_positive_negative.statistic,

            "p":
                ies_test_positive_negative.pvalue,
        },
    ]
)

ies_results_h3

Um IES menor na condição positiva indica melhor eficiência combinada.

## Decision rules

Hypothesis 3 is supported if positive emotion is associated with:

- significantly higher accuracy than both neutral and negative emotion;
- significantly faster reaction time without reduced accuracy;
- or significantly better processing efficiency relative to both
  comparison conditions.

Hypothesis 3 is partially supported if:

- positive emotion improves performance relative to only one comparator;
- or improvement is observed in only one outcome;
- or positive emotion produces faster responses without a statistically
  detectable accuracy advantage.

Hypothesis 3 is not supported if:

- positive emotion does not improve accuracy or reaction time relative
  to either comparator;
- or any apparent speed advantage is accompanied by a reliable loss of
  accuracy.

# Decisão programática

## 27. Extrair os valores principais

In [ ]:
acc_pn = (
    accuracy_positive_neutral_h3
    .iloc[0]
)

acc_pg = (
    accuracy_positive_negative_h3
    .iloc[0]
)

rt_pn = (
    rt_positive_neutral_h3
    .iloc[0]
)

rt_pg = (
    rt_positive_negative_h3
    .iloc[0]
)

acc_pg: representa Positive vs Negative.

## 28. Definir os padrões

In [ ]:
accuracy_better_neutral = (
    acc_pn["p"] < cfg.ALPHA
    and acc_pn["beta"] > 0
)

accuracy_better_negative = (
    acc_pg["p"] < cfg.ALPHA
    and acc_pg["beta"] > 0
)

rt_faster_neutral = (
    rt_pn["p"] < cfg.ALPHA
    and rt_pn["beta"] < 0
)

rt_faster_negative = (
    rt_pg["p"] < cfg.ALPHA
    and rt_pg["beta"] < 0
)

## 29. Avaliação da hipótese

In [ ]:
benefit_vs_neutral = (
    accuracy_better_neutral
    or rt_faster_neutral
)

benefit_vs_negative = (
    accuracy_better_negative
    or rt_faster_negative
)

if (
    benefit_vs_neutral
    and benefit_vs_negative
):
    h3_assessment = "supported"

elif (
    benefit_vs_neutral
    or benefit_vs_negative
):
    h3_assessment = "partially supported"

else:
    h3_assessment = "not supported"

In [ ]:
accuracy_supported = (
    accuracy_better_neutral
    or accuracy_better_negative
)

rt_supported = (
    rt_faster_neutral
    or rt_faster_negative
)

if (
    accuracy_better_neutral
    and accuracy_better_negative
):
    h3_assessment = "supported"

elif (
    rt_faster_neutral
    and rt_faster_negative
    and not accuracy_supported
):
    h3_assessment = "partially supported"

elif (
    benefit_vs_neutral
    or benefit_vs_negative
):
    h3_assessment = "partially supported"

else:
    h3_assessment = "not supported"

In [ ]:
print(
    "Hypothesis 3 assessment:",
    h3_assessment,
)

# Exportar resultados

## 30. Criar diretório

In [ ]:
h3_output_dir = (
    cfg.TABLES_DIR
    / "hypothesis_03"
)

h3_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

## 31. Salvar tabelas

In [ ]:
h3_sample.to_csv(
    h3_output_dir
    / "h3_sample.csv",
    index=False,
)

accuracy_summary_h3.to_csv(
    h3_output_dir
    / "h3_accuracy_descriptive.csv",
    index=False,
)

accuracy_contrasts_h3.to_csv(
    h3_output_dir
    / "h3_accuracy_models.csv",
    index=False,
)

rt_summary_h3.to_csv(
    h3_output_dir
    / "h3_rt_descriptive.csv",
    index=False,
)

rt_contrasts_h3.to_csv(
    h3_output_dir
    / "h3_rt_models.csv",
    index=False,
)

efficiency_summary_h3.to_csv(
    h3_output_dir
    / "h3_efficiency_descriptive.csv",
    index=False,
)

ies_results_h3.to_csv(
    h3_output_dir
    / "h3_efficiency_tests.csv",
    index=False,
)

## 32. Resumo da hipótese

In [ ]:
h3_summary = pd.DataFrame(
    {
        "hypothesis": [
            "H3"
        ],

        "participants": [
            h3_data[
                "participant_id"
            ].nunique()
        ],

        "accuracy_observations": [
            len(h3_data)
        ],

        "rt_observations": [
            len(rt_h3)
        ],

        "positive_neutral_accuracy_or": [
            acc_pn[
                "odds_ratio"
            ]
        ],

        "positive_neutral_accuracy_p": [
            acc_pn["p"]
        ],

        "positive_negative_accuracy_or": [
            acc_pg[
                "odds_ratio"
            ]
        ],

        "positive_negative_accuracy_p": [
            acc_pg["p"]
        ],

        "positive_neutral_rt_change": [
            rt_pn[
                "percent_change"
            ]
        ],

        "positive_neutral_rt_p": [
            rt_pn["p"]
        ],

        "positive_negative_rt_change": [
            rt_pg[
                "percent_change"
            ]
        ],

        "positive_negative_rt_p": [
            rt_pg["p"]
        ],

        "assessment": [
            h3_assessment
        ],
    }
)

h3_summary.to_csv(
    h3_output_dir
    / "h3_summary.csv",
    index=False,
)

h3_summary

# Última célula: resumo da analise

In [ ]:
positive_accuracy = (
    accuracy_summary_h3.loc[
        accuracy_summary_h3[
            "emotion_condition"
        ] == "Positive",
        "accuracy_mean",
    ]
    .iloc[0]
)

neutral_accuracy = (
    accuracy_summary_h3.loc[
        accuracy_summary_h3[
            "emotion_condition"
        ] == "Neutral",
        "accuracy_mean",
    ]
    .iloc[0]
)

negative_accuracy = (
    accuracy_summary_h3.loc[
        accuracy_summary_h3[
            "emotion_condition"
        ] == "Negative",
        "accuracy_mean",
    ]
    .iloc[0]
)

In [ ]:
print("=" * 75)
print("HYPOTHESIS 3 ANALYSIS SUMMARY")
print("=" * 75)

print(
    f"Participants: "
    f"{h3_data['participant_id'].nunique()}"
)

print(
    f"Accuracy observations: "
    f"{len(h3_data)}"
)

print(
    f"RT observations: "
    f"{len(rt_h3)}"
)

print()

print(
    f"Positive accuracy: "
    f"{positive_accuracy:.3f}"
)

print(
    f"Neutral accuracy: "
    f"{neutral_accuracy:.3f}"
)

print(
    f"Negative accuracy: "
    f"{negative_accuracy:.3f}"
)

print()

print(
    "Positive vs Neutral accuracy OR: "
    f"{acc_pn['odds_ratio']:.3f}"
)

print(
    "Positive vs Neutral accuracy p: "
    f"{acc_pn['p']:.4f}"
)

print(
    "Positive vs Negative accuracy OR: "
    f"{acc_pg['odds_ratio']:.3f}"
)

print(
    "Positive vs Negative accuracy p: "
    f"{acc_pg['p']:.4f}"
)

print()

print(
    "Positive vs Neutral RT change: "
    f"{rt_pn['percent_change']:.1f}%"
)

print(
    "Positive vs Neutral RT p: "
    f"{rt_pn['p']:.4f}"
)

print(
    "Positive vs Negative RT change: "
    f"{rt_pg['percent_change']:.1f}%"
)

print(
    "Positive vs Negative RT p: "
    f"{rt_pg['p']:.4f}"
)

print()

print(
    f"H3 assessment: "
    f"{h3_assessment}"
)

print("=" * 75)